In [ ]:
# --- Parameters (override at run time, e.g. via Data Factory / scheduler) ----
# Tag this cell as `parameters` in the notebook UI if you want papermill-style
# overrides; the variables defined here can also just be edited inline.

TENANT_NAME      = "UPDC"                 # case-insensitive match
ASSESSMENT_TYPE  = "Security Assessment"  # 'Security Assessment' | 'Copilot Readiness' | 'Copilot Assessment'
OUTPUT_DIR       = "reports"              # under Files/<OUTPUT_DIR>/<tenant>/...
TEMPLATE_PATH    = "/lakehouse/default/Files/templates/assessment_report.md.j2"
TOP_FINDINGS_N   = 10                     # how many top failures to surface in the body
LOW_SCORE_CUTOFF = 60                     # triggers the Managed Security Service rule


StatementMeta(, 750e2787-0c75-416e-8dab-42694d86eef9, 14, Finished, Available, Finished, False)

# Generate Assessment Report

**Default lakehouse:** `ManagedServiceData`

Renders a Markdown report for a single tenant from a Jinja2 template.

## Inputs
- `TENANT_NAME`  e.g. `"UPDC"` (case-insensitive)
- `ASSESSMENT_TYPE`  one of `"Security Assessment"`, `"Copilot Readiness"`, `"Copilot Assessment"`
- `OUTPUT_DIR`  lakehouse Files subfolder
- `TEMPLATE_PATH`  Jinja2 template; auto-seeded on first run if missing

## Data sources
- `<type>_reports`  header (score, pass/fail/warn counts, dates)
- `<type>_checks`   per-check failures used for evidence + catalog matching
- `<type>_categories`  category-level scores
- `recommendation_catalog`  SKU recommendations (denormalized, single-token LIKE)

## Output
`Files/reports/<tenant-slug>/<YYYY-MM-DD>__<assessment-type-slug>.md`

The template lives in the lakehouse so it can be edited without touching the notebook.


In [2]:
%pip install jinja2 --quiet


StatementMeta(, 750e2787-0c75-416e-8dab-42694d86eef9, 12, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [4]:
# Imports & helpers
import os, re
from datetime import date
from pyspark.sql import functions as F
from jinja2 import Environment, BaseLoader

SCHEMA  = "dbo"

def rd(t):
    return spark.table(f"{SCHEMA}.{t}")

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

# Resolve assessment family tag from ASSESSMENT_TYPE param
FAMILY_MAP = {
    "security assessment": "Security",
    "copilot readiness":   "CopilotReadiness",
    "copilot assessment":  "CopilotAssessment",
}
FAMILY = FAMILY_MAP.get(ASSESSMENT_TYPE.lower(), "Security")

# Derive template + output paths from the default lakehouse mount
_LH_FILES  = "/lakehouse/default/Files"
_TEMPLATE_PATH = f"{_LH_FILES}/templates/assessment_report.md.j2"

print(f"Tenant: {TENANT_NAME!r}  |  Family: {FAMILY}  |  Output: Files/{OUTPUT_DIR}/")


StatementMeta(, 750e2787-0c75-416e-8dab-42694d86eef9, 15, Finished, Available, Finished, False)

Tenant: 'UPDC'  |  Family: Security  |  Output: Files/reports/


In [5]:
# Seed Jinja2 template — write to lakehouse Files/templates/ if not already present
TEMPLATE_CONTENT = """\
# {{ assessment_type }} Report — {{ tenant_name }}

**Date:** {{ report_date }}
**Assessment ID:** {{ assessment_id }}
**Assessment Date:** {{ assessment_date }}

---

## Executive Summary

| Metric | Value |
|--------|-------|
| Overall Score | **{{ overall_score }}%** |
| Passed Checks | {{ passed }} |
| Failed Checks | {{ failed }} |
| Warnings | {{ warnings }} |
| Security Exposure | {{ exposure }}% |
| Enablement Opportunity | {{ enablement_opportunity }} |

{% if has_pc %}
**Partner Center:** {{ pc_company_name }} · {{ pc_csp_relationship }} · {{ pc_country }}
{% endif %}

---

## Top {{ findings | length }} Findings

{% for f in findings %}
### {{ loop.index }}. {{ f.check_name }}
- **Category:** {{ f.category }}{% if f.subcategory %} / {{ f.subcategory }}{% endif %}
- **Status:** {{ f.status }} | **Priority:** {{ f.priority }}
- **Rationale:** {{ f.business_rationale or "—" }}

{% endfor %}

---

## Recommendations

{% if recommendations %}
| SKU | Type | Impact | Supporting Findings | Talking Point |
|-----|------|--------|-------------------|---------------|
{% for r in recommendations %}
| {{ r.recommendation_sku }} | {{ r.recommendation_type }} | {{ r.impact_weight }} | {{ r.matched_issue_count }} | {{ r.talking_point }} |
{% endfor %}
{% else %}
No catalog matches found for the current failed checks.
{% endif %}

---

## Category Scores

{% for c in categories %}
- **{{ c.category }}**: {{ c.failed }} failed, {{ c.warnings }} warnings
{% endfor %}

---

*Generated by ManagedService-X · {{ report_date }}*
"""

os.makedirs(f"{_LH_FILES}/templates", exist_ok=True)
if not os.path.exists(_TEMPLATE_PATH):
    with open(_TEMPLATE_PATH, "w") as fh:
        fh.write(TEMPLATE_CONTENT)
    print(f"✓ Template seeded: {_TEMPLATE_PATH}")
else:
    print(f"✓ Template already exists: {_TEMPLATE_PATH}")


StatementMeta(, 750e2787-0c75-416e-8dab-42694d86eef9, 16, Finished, Available, Finished, False)

✓ Template already exists: /lakehouse/default/Files/templates/assessment_report.md.j2


In [6]:
# Fetch all data for the tenant from the semantic layer
tenant_key = TENANT_NAME.lower().strip()

# ── Customer row ─────────────────────────────────────────────────────────────
cust_row = (rd("sem_dim_customer")
    .filter(F.col("customer_key").contains(tenant_key) | F.lower(F.col("tenant_name")).contains(tenant_key))
    .orderBy(F.length("tenant_name"))
    .limit(1)
    .collect())

if not cust_row:
    raise ValueError(f"No customer found matching {TENANT_NAME!r}. Check TENANT_NAME.")

cust = cust_row[0]
resolved_key = cust["customer_key"]
print(f"Resolved tenant: {cust['tenant_name']!r}  (key={resolved_key!r})")

# ── Posture row ───────────────────────────────────────────────────────────────
posture = (rd("sem_fact_posture")
    .filter(F.col("customer_key") == resolved_key)
    .collect())
posture = posture[0] if posture else None

# ── Latest assessment for the requested family ────────────────────────────────
asmt = (rd("sem_fact_assessment")
    .filter((F.col("customer_key") == resolved_key) & (F.col("assessment_family") == FAMILY))
    .orderBy(F.col("assessment_date_parsed").desc_nulls_last())
    .limit(1)
    .collect())

if not asmt:
    raise ValueError(f"No {FAMILY!r} assessment found for tenant {cust['tenant_name']!r}.")

asmt = asmt[0]
print(f"Assessment: {asmt['assessment_id']}  date={asmt['assessment_date']}  score={asmt['overall_score_pct']}")

# ── Top failed/warning checks ────────────────────────────────────────────────
priority_order = F.when(F.col("priority") == "Critical", 1)\
    .when(F.col("priority") == "High", 2)\
    .when(F.col("priority") == "Medium", 3)\
    .otherwise(4)

findings = (rd("sem_fact_check")
    .filter(
        (F.col("customer_key") == resolved_key) &
        (F.col("check_family") == FAMILY) &
        F.col("is_latest_assessment") &
        F.col("is_issue")
    )
    .orderBy(priority_order, F.col("status"))
    .limit(TOP_FINDINGS_N)
    .collect())

print(f"Top findings: {len(findings)} rows")

# ── Category breakdown ────────────────────────────────────────────────────────
categories = (rd("sem_fact_check")
    .filter(
        (F.col("customer_key") == resolved_key) &
        (F.col("check_family") == FAMILY) &
        F.col("is_latest_assessment")
    )
    .groupBy("category")
    .agg(
        F.sum(F.when(F.col("status") == "Failed", 1).otherwise(0)).alias("failed"),
        F.sum(F.when(F.col("status") == "Warning", 1).otherwise(0)).alias("warnings"),
    )
    .orderBy(F.col("failed").desc())
    .collect())

# ── Recommendations ────────────────────────────────────────────────────────────
recs = (rd("sem_fact_recommendation")
    .filter(F.col("customer_key") == resolved_key)
    .orderBy(F.col("matched_issue_count").desc())
    .collect())

print(f"Recommendations: {len(recs)} rows")
print(f"Categories: {len(categories)} rows")


StatementMeta(, 750e2787-0c75-416e-8dab-42694d86eef9, 17, Finished, Available, Finished, False)

Resolved tenant: 'UPDC'  (key='updc')
Assessment: 4c764a8fcffc8a6a  date=2026-06-04  score=21.0
Top findings: 10 rows
Recommendations: 18 rows
Categories: 8 rows


In [7]:
# Render Jinja2 template and save to lakehouse Files
with open(_TEMPLATE_PATH) as fh:
    template_src = fh.read()

env = Environment(loader=BaseLoader(), autoescape=False)
template = env.from_string(template_src)

score_raw = asmt["overall_score_pct"]
try:
    overall_score = round(float(score_raw), 1) if score_raw is not None else "N/A"
except (TypeError, ValueError):
    overall_score = score_raw

exposure = None
if posture and posture["security_exposure"] is not None:
    try:
        exposure = round(float(posture["security_exposure"]), 1)
    except (TypeError, ValueError):
        pass

ctx = {
    "tenant_name":            cust["tenant_name"],
    "assessment_type":        ASSESSMENT_TYPE,
    "report_date":            str(date.today()),
    "assessment_id":          asmt["assessment_id"],
    "assessment_date":        asmt["assessment_date"],
    "overall_score":          overall_score,
    "passed":                 asmt["passed_count"],
    "failed":                 asmt["failed_count"],
    "warnings":               asmt["warnings_count"],
    "exposure":               exposure or "N/A",
    "enablement_opportunity": posture["enablement_opportunity"] if posture else "Unknown",
    "has_pc":                 bool(cust["pc_tenant_id"]),
    "pc_company_name":        cust["pc_company_name"] or "",
    "pc_csp_relationship":    cust["pc_csp_relationship"] or "",
    "pc_country":             cust["pc_country"] or "",
    "findings":               [row.asDict() for row in findings],
    "recommendations":        [row.asDict() for row in recs],
    "categories":             [row.asDict() for row in categories],
}

rendered = template.render(**ctx)

# Write output to Files/<OUTPUT_DIR>/<tenant-slug>/
tenant_slug = slugify(cust["tenant_name"])
type_slug   = slugify(ASSESSMENT_TYPE)
out_dir     = f"{_LH_FILES}/{OUTPUT_DIR}/{tenant_slug}"
os.makedirs(out_dir, exist_ok=True)
out_path    = f"{out_dir}/{date.today()}__{type_slug}.md"

with open(out_path, "w", encoding="utf-8") as fh:
    fh.write(rendered)

print(f"✓ Report saved: Files/{OUTPUT_DIR}/{tenant_slug}/{date.today()}__{type_slug}.md")
print(f"  {len(rendered):,} characters  |  {len(findings)} findings  |  {len(recs)} recommendations")
print()
print(rendered[:2000], "..." if len(rendered) > 2000 else "")


StatementMeta(, 750e2787-0c75-416e-8dab-42694d86eef9, 18, Finished, Available, Finished, False)

✓ Report saved: Files/reports/updc/2026-07-11__security_assessment.md
  9,391 characters  |  10 findings  |  18 recommendations

# Security Assessment Report — UPDC

**Date:** 2026-07-11
**Assessment ID:** 4c764a8fcffc8a6a
**Assessment Date:** 2026-06-04

---

## Executive Summary

| Metric | Value |
|--------|-------|
| Overall Score | **21.0%** |
| Passed Checks | 21 |
| Failed Checks | 84 |
| Warnings | 1 |
| Security Exposure | N/A% |
| Enablement Opportunity | High |



---

## Top 10 Findings


### 1. Ensure a managed device is required for authentication Conditional Access
- **Category:** Identity & Access Management / Entra
- **Status:** Failed | **Priority:** Medium
- **Rationale:** "Managed" devices are considered more secure because they often have additional configuration hardening enforced through centralized management such as Intune or Group Policy. These devices are also typically equipped with MDR/EDR, managed patching and alerting systems. As a result, they provide a 